This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

## Task

A table of the following form is exported from ClickHouse:
```
   user_id  item_id  qty  price
0     8249        4    1  425.0
1    34150        3    1  504.0
2    51564        6    3  744.0
3    55459        7    2  624.0
4    67319        9    1  533.0
```
This is a table showing how many of each item a user bought over a given period of time.

The task is to implement a `UserItemMatrix` class that takes purchase data as input and builds a sparse User-Item matrix.

The User-Item matrix contains:
- rows representing users
- columns representing items

The intersection of a user and an item is the number of items X that user Y bought.

## Solution

The matrix is stored as a `scipy.sparse.csr_matrix`.

The `UserItemMatrix` class exposes five methods that return information about the matrix. They are wrapped in `@property`, so all the attributes are computed once in `__init__`.

In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
from typing import Dict
import numpy as np

class UserItemMatrix:
    def __init__(self, sales_data: pd.DataFrame):
        """Class initialization. You can make necessary
        calculations here.

        Args:
            sales_data (pd.DataFrame): Sales dataset.

        Example:
            sales_data (pd.DataFrame):

                user_id  item_id  qty   price
            0        1      118    1   626.66
            1        1      285    1  1016.57
            2        2     1229    3   518.99
            3        4     1688    2   940.84
            4        5     2068    1   571.36
            ...

        """
        self._sales_data = sales_data.copy()

        self._user_count = self._sales_data.user_id.nunique()
        self._item_count = self._sales_data.item_id.nunique()

        self._user_map = {
            val: idx
            for idx, val in enumerate(self._sales_data.user_id.unique())
        }

        self._item_map = {
            val: idx
            for idx, val in enumerate(np.sort(self._sales_data.item_id.unique()))
        }

        self._sales_data["user_idx"] = self._sales_data.user_id. \
                                       map(lambda x: self._user_map[x])
        self._sales_data["item_idx"] = self._sales_data.item_id. \
                                       map(lambda x: self._item_map[x])

        data = self._sales_data.qty.to_numpy()
        row = self._sales_data.user_idx.to_numpy()
        col = self._sales_data.item_idx.to_numpy()

        self._matrix = csr_matrix((data, (row, col)))

    @property
    def user_count(self) -> int:
        """
        Returns:
            int: the number of users in sales_data.
        """
        return self._user_count

    @property
    def item_count(self) -> int:
        """
        Returns:
            int: the number of items in sales_data.
        """
        return self._item_count

    @property
    def user_map(self) -> Dict[int, int]:
        """Creates a mapping from user_id to matrix rows indexes.

        Example:
            sales_data (pd.DataFrame):

                user_id  item_id  qty   price
            0        1      118    1   626.66
            1        1      285    1  1016.57
            2        2     1229    3   518.99
            3        4     1688    2   940.84
            4        5     2068    1   571.36

            user_map (Dict[int, int]):
                {1: 0, 2: 1, 4: 2, 5: 3}

        Returns:
            Dict[int, int]: User map
        """
        return self._user_map

    @property
    def item_map(self) -> Dict[int, int]:
        """Creates a mapping from item_id to matrix rows indexes.

        Example:
            sales_data (pd.DataFrame):

                user_id  item_id  qty   price
            0        1      118    1   626.66
            1        1      285    1  1016.57
            2        2     1229    3   518.99
            3        4     1688    2   940.84
            4        5     2068    1   571.36

            item_map (Dict[int, int]):
                {118: 0, 285: 1, 1229: 2, 1688: 3, 2068: 4}

        Returns:
            Dict[int, int]: Item map
        """
        return self._item_map

    @property
    def csr_matrix(self) -> csr_matrix:
        """User items matrix in form of CSR matrix.

        User row_ind, col_ind as
        rows and cols indecies(mapped from user/item map).

        Returns:
            csr_matrix: CSR matrix
        """
        return self._matrix

## Normalization

I implement four ways to normalize the matrix: by row, by column, TF-IDF, and BM-25.

In [ ]:
from scipy.sparse import csr_matrix


class Normalization:
    @staticmethod
    def by_column(matrix: csr_matrix) -> csr_matrix:
        """Normalization by column

        Args:
            matrix (csr_matrix): User-Item matrix of size (N, M)

        Returns:
            csr_matrix: Normalized matrix of size (N, M)
        """
        sum_by_product = np.power(matrix.sum(axis=0).astype(float), -1)
        norm_matrix = matrix.multiply(sum_by_product)

        return norm_matrix.tocsr()

    @staticmethod
    def by_row(matrix: csr_matrix) -> csr_matrix:
        """Normalization by row

        Args:
            matrix (csr_matrix): User-Item matrix of size (N, M)

        Returns:
            csr_matrix: Normalized matrix of size (N, M)
        """
        sum_by_user = np.power(matrix.sum(axis=1).astype(float), -1)
        norm_matrix = matrix.multiply(sum_by_user)

        return norm_matrix.tocsr()

    @staticmethod
    def tf_idf(matrix: csr_matrix) -> csr_matrix:
        """Normalization using tf-idf

        Args:
            matrix (csr_matrix): User-Item matrix of size (N, M)

        Returns:
            csr_matrix: Normalized matrix of size (N, M)
        """
        
        tf = Normalization.by_row(matrix)
        
        number_of_users = tf.shape[0]
        nonzero_users = (tf != 0).sum(axis=0)
        idf = np.log(number_of_users * np.power(nonzero_users.astype(float), -1))
        
        norm_matrix = tf.multiply(idf)
        
        return norm_matrix.tocsr()

    @staticmethod
    def bm_25(
        matrix: csr_matrix, k1: float = 2.0, b: float = 0.75
    ) -> csr_matrix:
        """Normalization based on BM-25

        Args:
            matrix (csr_matrix): User-Item matrix of size (N, M)

        Returns:
            csr_matrix: Normalized matrix of size (N, M)
        """
        tf = Normalization.by_row(matrix)
        idf_tf = Normalization.tf_idf(matrix)
        
        avgdl = np.mean(matrix.sum(axis=1))
        
        delta = matrix.copy()
        delta.data = np.multiply(delta.data, np.power(tf.data, -1))
        delta.data = tf.data + k1 * (1 - b + b / avgdl * delta.data)

        norm_matrix = idf_tf * (k1 + 1)
        
        norm_matrix.data = np.multiply(norm_matrix.data, np.power(delta.data, -1))
        
        return norm_matrix.tocsr()


## Item embeddings

To factorize the User-Item matrix I use Alternating Least Squares (ALS), an algorithm widely used in recommender systems. The matrix is factorized into user and item embeddings, and the function returns the item embedding matrix (its row order matches the order of items in the input).

In [ ]:
import implicit

def items_embeddings(ui_matrix: csr_matrix, dim: int) -> np.ndarray:
    """Build items embedding using factorization model.
    The order of items should be the same in the output matrix.

    Args:
        ui_matrix (csr_matrix): User-Item matrix of size (N, M)
        dim (int): Dimention of embedding vectors

    Returns:
        np.ndarray: Items embeddings matrix of size (M, dim)
    """
    
    als = implicit.cpu.als.AlternatingLeastSquares(factors=dim, iterations=10)
    als.fit(ui_matrix)
    
    items_vec = als.item_factors
    
    return items_vec
